# Task 2: Production-Grade RAG Pipeline from Scratch
## Evaluation over NIST AI 100-1 (AI Risk Management Framework 1.0 - 48 Pages)

### Strict Constraint Notice:
**ZERO Frameworks Used**: This notebook strictly avoids LangChain, LlamaIndex, Haystack, or any high-level wrapper. Every single component—PDF extraction, chunking, indexing, dense retrieval, BM25 keyword scoring, reciprocal rank fusion, cross-encoder reranking, and LLM answer synthesis—is engineered in native Python with PyMuPDF/pypdf, FAISS, Sentence Transformers, rank_bm25, and Transformers.

---

### Complete RAG Pipeline Architecture:
```
  [Public 48-Page PDF: NIST.AI.100-1.pdf]
                    │
                    ▼
       1. Page-by-Page PDF Extraction
         (Preserve metadata & page #)
                    │
                    ▼
          2. Cleaning & Normalization
                    │
         ┌──────────┴──────────┐
         ▼                     ▼
   Config A (300t/50ov)   Config B (500t/50ov)
         │                     │
         ▼                     ▼
  Dense Embeddings      Dense Embeddings
(all-MiniLM-L6-v2)     (all-MiniLM-L6-v2)
         │                     │
         ▼                     ▼
    FAISS Index           FAISS Index
         │                     │
         ├─────────────────────┤
         ▼                     ▼
     [Dense Retrieval]     [BM25 Keyword Retrieval]
         │                     │
         └──────────┬──────────┘
                    ▼
          3. Hybrid Reciprocal Rank Fusion (RRF)
                    │
                    ▼
       4. Cross-Encoder Reranker
         (cross-encoder/ms-marco-MiniLM-L-6-v2)
                    │
                    ▼
        5. LLM Answer Generation with Strict Attribution
         (Qwen/Qwen2.5-0.5B-Instruct)
```



### 1. Environment Setup & Dependency Installation
**What it does:** Installs foundational retrieval libraries (`pypdf`, `pymupdf`, `sentence-transformers`, `faiss-cpu`, `rank-bm25`, `transformers`, `torch`).
**Why it is required:** Sets up our bespoke retrieval stack without any high-level abstractions like LangChain.
**Expected Output:** Clean installation with zero version conflicts.



In [1]:
!pip install -q --upgrade pip
!pip install -q pypdf pymupdf sentence-transformers faiss-cpu rank-bm25 transformers accelerate matplotlib seaborn pandas



### 2. Device Detection & Workspace Directory Setup
**What it does:** Detects GPU availability for sentence embedding and LLM generation. Creates directory structure for documents, FAISS indexes, and benchmark results.



In [2]:
import os
import sys
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Compute Device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

BASE_DIR = "/content/LLM-Engineering-Assignment/Task-2-RAG" if os.path.exists("/content") else "./Task-2-RAG"
DOCS_DIR = os.path.join(BASE_DIR, "documents")
INDEX_DIR = os.path.join(BASE_DIR, "indexes")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

for d in [DOCS_DIR, INDEX_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Task 2 Workspace initialized at: {BASE_DIR}")



Active Compute Device: cuda
GPU Model: Tesla T4
Total VRAM: 14.56 GB
Task 2 Workspace initialized at: /content/LLM-Engineering-Assignment/Task-2-RAG


### 3. Document Ingestion & Integrity Validation
**Document Selected:** National Institute of Standards and Technology (NIST) Special Publication: *Artificial Intelligence Risk Management Framework (AI RMF 1.0)*, NIST AI 100-1.
- **Source URL:** `https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf`
- **Page Count:** 48 pages ($\ge 30$ pages requirement verified).
- **License / Usage:** Public domain technical standard published by the U.S. Federal Government.
**Validation:** Checks HTTP response status and verifies exact PDF page count before proceeding.



In [3]:
import urllib.request
import pypdf

PDF_URL = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
PDF_PATH = os.path.join(DOCS_DIR, "NIST.AI.100-1.pdf")

print(f"Downloading technical document from: {PDF_URL}")
try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(PDF_URL, headers=headers)
    with urllib.request.urlopen(req) as response, open(PDF_PATH, 'wb') as out_file:
        out_file.write(response.read())
    print("Download completed successfully.")
except Exception as e:
    print(f"Download failed: {e}. Please ensure network access or manually upload NIST.AI.100-1.pdf.")

# Rigorous Page Count & Integrity Validation
assert os.path.exists(PDF_PATH), f"PDF file not found at {PDF_PATH}"
reader = pypdf.PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f"\n--- PDF Validation Check ---")
print(f"Document File: {os.path.basename(PDF_PATH)}")
print(f"Total Pages Count: {total_pages}")
assert total_pages >= 30, f"Error: Document has {total_pages} pages, which is less than the required 30 pages!"
print("Document verification PASSED (>= 30 pages verified).")



Download completed successfully.

--- PDF Validation Check ---
Document File: NIST.AI.100-1.pdf
Total Pages Count: 48
Document verification PASSED (>= 30 pages verified).


### 4. Page-by-Page Text Extraction & Metadata Preservation
**What it does:** Extracts text sequentially page-by-page. Creates structured records containing `page_number`, `raw_text`, and `clean_text`.
**Why it is required:** Precise source attribution in RAG requires tracking exactly which page each passage originated from.
**What can go wrong:** Header/footer noise, blank pages, and Unicode formatting artifacts. We clean whitespace while keeping sentence structure intact.



In [4]:
import re

page_records = []
for idx, page in enumerate(reader.pages):
    page_num = idx + 1
    raw = page.extract_text() or ""

    # Text cleaning: normalize whitespace, remove carriage returns, strip excessive linebreaks
    cleaned = re.sub(r'\r\n', '\n', raw)
    cleaned = re.sub(r'[ \t]+', ' ', cleaned)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()

    if len(cleaned) > 50:  # Filter out blank/trivial pages
        page_records.append({
            "doc_name": "NIST.AI.100-1.pdf",
            "page_number": page_num,
            "text": cleaned,
            "char_count": len(cleaned)
        })

print(f"Successfully extracted text from {len(page_records)} active pages.")
print(f"Sample page 20 preview (First 300 chars):\n{page_records[19]['text'][:300]}...")



Successfully extracted text from 48 active pages.
Sample page 20 preview (First 300 chars):
NIST AI 100-1 AI RMF 1.0
Employing safety considerations during the lifecycle and starting as early as possible with
planning and design can prevent failures or conditions that can render a system dangerous.
Other practical approaches for AI safety often relate to rigorous simulation and in-domain
t...


### 5. Chunking Experiment: 300 Tokens vs. 500 Tokens
**Comparative Design:**
- **Configuration A**: Chunk size = 300 tokens, Overlap = 50 tokens.
- **Configuration B**: Chunk size = 500 tokens, Overlap = 50 tokens.
**Tokenizer:** Uses the Hugging Face `sentence-transformers/all-MiniLM-L6-v2` tokenizer to measure exact token lengths.
**Metadata Preservation:** Every chunk carries `chunk_id`, `doc_name`, `page_number`, `token_count`, and `text`.



In [5]:
from transformers import AutoTokenizer

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

def chunk_document(pages, chunk_size=300, overlap=50):
    chunks = []
    chunk_counter = 0

    for page in pages:
        p_text = page["text"]
        p_num = page["page_number"]
        tokens = embed_tokenizer.encode(p_text, add_special_tokens=False)

        start = 0
        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunk_tokens = tokens[start:end]
            chunk_text = embed_tokenizer.decode(chunk_tokens)

            chunk_counter += 1
            chunks.append({
                "chunk_id": f"chunk_sz{chunk_size}_{chunk_counter:04d}",
                "doc_name": page["doc_name"],
                "page_number": p_num,
                "text": chunk_text,
                "token_count": len(chunk_tokens)
            })

            if end == len(tokens):
                break
            start += (chunk_size - overlap)

    return chunks

chunks_300 = chunk_document(page_records, chunk_size=300, overlap=50)
chunks_500 = chunk_document(page_records, chunk_size=500, overlap=50)

print(f"--- Chunking Experiment Statistics ---")
print(f"Config A (300 tokens, 50 overlap): {len(chunks_300)} total chunks")
print(f"Config B (500 tokens, 50 overlap): {len(chunks_500)} total chunks")



config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (624 > 512). Running this sequence through the model will result in indexing errors


--- Chunking Experiment Statistics ---
Config A (300 tokens, 50 overlap): 97 total chunks
Config B (500 tokens, 50 overlap): 70 total chunks


### 6. Vector Embedding Generation & FAISS Cosine Indexing
**Model:** `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
**Cosine Similarity Implementation:** Embeddings are L2-normalized ($||v||_2 = 1$). We build an `IndexFlatIP` (Inner Product) FAISS index. For unit vectors, the inner product is mathematically identical to cosine similarity:
$$\text{CosineSim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|} = u \cdot v$$



In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL_ID, device=device)

def build_faiss_index(chunks):
    texts = [c["text"] for c in chunks]
    print(f"Computing embeddings for {len(texts)} chunks...")
    embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    embeddings = np.ascontiguousarray(embeddings.astype(np.float32))

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index, embeddings

print("Building FAISS index for 300-token chunks...")
index_300, embeds_300 = build_faiss_index(chunks_300)

print("Building FAISS index for 500-token chunks...")
index_500, embeds_500 = build_faiss_index(chunks_500)

# Persist indexes
faiss.write_index(index_300, os.path.join(INDEX_DIR, "faiss_index_300.bin"))
faiss.write_index(index_500, os.path.join(INDEX_DIR, "faiss_index_500.bin"))
print("Both FAISS indexes built and saved to disk.")



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index for 300-token chunks...
Computing embeddings for 97 chunks...
Building FAISS index for 500-token chunks...
Computing embeddings for 70 chunks...
Both FAISS indexes built and saved to disk.


### 7. BM25 Keyword Retrieval & Hybrid Reciprocal Rank Fusion (RRF)
**What it does:**
1. **BM25 Keyword Search**: Tokenizes documents with whitespace and lowercase normalization; computes BM25 relevance scores using `rank_bm25.BM25Okapi`.
2. **Dense Vector Search**: Computes top-K FAISS inner-product nearest neighbors.
3. **Reciprocal Rank Fusion (RRF)**: Merges sparse keyword ranks and dense vector ranks using:
$$RRF(d) = \sum_{m \in \{dense, bm25\}} \frac{1}{k + \text{rank}_m(d)}$$
where $k = 60$ is the standard smoothing constant.



In [7]:
from rank_bm25 import BM25Okapi

class HybridRetriever:
    def __init__(self, chunks, faiss_index, embed_model):
        self.chunks = chunks
        self.faiss_index = faiss_index
        self.embed_model = embed_model

        # Tokenize for BM25
        self.tokenized_corpus = [c["text"].lower().split() for c in chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def dense_search(self, query, top_k=10):
        q_emb = self.embed_model.encode([query], normalize_embeddings=True)
        q_emb = np.ascontiguousarray(q_emb.astype(np.float32))
        scores, indices = self.faiss_index.search(q_emb, top_k)
        results = []
        for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
            c = dict(self.chunks[idx])
            c["dense_score"] = float(score)
            c["dense_rank"] = rank + 1
            results.append(c)
        return results

    def bm25_search(self, query, top_k=10):
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for rank, idx in enumerate(top_indices):
            c = dict(self.chunks[idx])
            c["bm25_score"] = float(scores[idx])
            c["bm25_rank"] = rank + 1
            results.append(c)
        return results

    def hybrid_search(self, query, top_k=10, rrf_k=60):
        dense_res = self.dense_search(query, top_k=top_k*2)
        bm25_res = self.bm25_search(query, top_k=top_k*2)

        rrf_scores = {}
        chunk_map = {}

        for r, item in enumerate(dense_res):
            cid = item["chunk_id"]
            chunk_map[cid] = item
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + (r + 1)))

        for r, item in enumerate(bm25_res):
            cid = item["chunk_id"]
            if cid not in chunk_map:
                chunk_map[cid] = item
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + (r + 1)))

        sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        final_results = []
        for cid, score in sorted_chunks:
            c = dict(chunk_map[cid])
            c["rrf_score"] = score
            final_results.append(c)
        return final_results

retriever_300 = HybridRetriever(chunks_300, index_300, embed_model)
retriever_500 = HybridRetriever(chunks_500, index_500, embed_model)
print("Hybrid Retrievers for Config 300 and Config 500 initialized.")



Hybrid Retrievers for Config 300 and Config 500 initialized.


### 8. Cross-Encoder Reranking
**What it does:** Re-evaluates top retrieved passages using `cross-encoder/ms-marco-MiniLM-L-6-v2`.
**Why it is required:** Bi-encoders (dense FAISS) compute query and passage embeddings independently (dot product between vectors). A Cross-Encoder performs full cross-attention over query-passage pairs:
$$\text{Score} = \text{CrossEncoder}(\text{Query}, \text{Passage})$$
capturing rich token-level lexical and semantic interactions.



In [8]:
from sentence_transformers import CrossEncoder

RERANKER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_ID, device=device)

def rerank_passages(query, passages, top_k=5):
    pairs = [[query, p["text"]] for p in passages]
    scores = reranker.predict(pairs)
    for p, score in zip(passages, scores):
        p["reranker_score"] = float(score)
    ranked = sorted(passages, key=lambda x: x["reranker_score"], reverse=True)[:top_k]
    return ranked

print("Cross-Encoder reranker initialized.")



config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder reranker initialized.


### 9. Hand-Curated 15-Question Benchmark Dataset
**What it does:** Loads 15 factual questions specifically authored against NIST AI 100-1 with verified ground-truth page numbers and expected passages.



In [13]:
import json

# Load verified evaluation benchmark
eval_dataset_path = os.path.join(BASE_DIR, "eval_questions.json")
if not os.path.exists(eval_dataset_path):
    # Fallback to local repo location
    eval_dataset_path = "/content/eval_questions.json"

with open(eval_dataset_path, "r", encoding="utf-8") as f:
    eval_questions = json.load(f)

print(f"Loaded {len(eval_questions)} verified evaluation benchmark questions.")
print(f"Sample Question 1: '{eval_questions[0]['question']}' -> Target Page: {eval_questions[0]['expected_page']}")



Loaded 15 verified evaluation benchmark questions.
Sample Question 1: 'What are the four core functions of the NIST AI Risk Management Framework Core?' -> Target Page: 20


### 10. Quantitative Retrieval Benchmarking
**Formulas:**
- $\text{Recall@K} = \frac{\sum_{i=1}^{Q} \mathbb{I}(\text{relevant chunk in top } K)}{Q}$
- $\text{MRR} = \frac{1}{Q} \sum_{i=1}^{Q} \frac{1}{\text{rank of first relevant chunk}}$
- **Relevance Criterion:** A retrieved chunk is deemed relevant if its page number matches the expected ground-truth page ($\pm 1$ adjacent page due to chunk boundary overlap) AND contains core key terms.



In [14]:
import time
import pandas as pd

def is_relevant(chunk, q_item):
    exp_page = q_item["expected_page"]
    # Check page proximity (exact or adjacent page transition)
    page_match = abs(chunk["page_number"] - exp_page) <= 1
    # Check key term containment
    text_lower = chunk["text"].lower()
    terms_found = sum(1 for term in q_item.get("key_terms", []) if term.lower() in text_lower)
    return page_match and (terms_found >= 1)

def evaluate_retrieval_system(retriever, questions, top_k=5, use_reranker=True):
    r1_hits = 0
    r5_hits = 0
    mrr_sum = 0.0
    latencies = []

    detailed_results = []

    for q in questions:
        t0 = time.time()
        # Stage 1: Hybrid retrieval
        candidates = retriever.hybrid_search(q["question"], top_k=top_k*2)
        # Stage 2: Cross-Encoder reranking
        if use_reranker:
            final_chunks = rerank_passages(q["question"], candidates, top_k=top_k)
        else:
            final_chunks = candidates[:top_k]
        latency_ms = (time.time() - t0) * 1000.0
        latencies.append(latency_ms)

        # Determine first relevant rank
        hit_rank = None
        for r_idx, c in enumerate(final_chunks):
            if is_relevant(c, q):
                hit_rank = r_idx + 1
                break

        if hit_rank == 1:
            r1_hits += 1
        if hit_rank is not None and hit_rank <= 5:
            r5_hits += 1
            mrr_sum += (1.0 / hit_rank)

        detailed_results.append({
            "id": q["id"],
            "question": q["question"],
            "target_page": q["expected_page"],
            "hit_rank": hit_rank,
            "latency_ms": latency_ms,
            "top_retrieved_page": final_chunks[0]["page_number"] if final_chunks else None,
            "top_retrieved_text": final_chunks[0]["text"][:150] if final_chunks else ""
        })

    num_q = len(questions)
    metrics = {
        "Recall@1 (%)": round((r1_hits / num_q) * 100, 2),
        "Recall@5 (%)": round((r5_hits / num_q) * 100, 2),
        "MRR": round(mrr_sum / num_q, 4),
        "Avg Latency (ms)": round(float(np.mean(latencies)), 2)
    }
    return metrics, detailed_results

print("Evaluating 300-Token Chunk Configuration...")
metrics_300, details_300 = evaluate_retrieval_system(retriever_300, eval_questions)

print("Evaluating 500-Token Chunk Configuration...")
metrics_500, details_500 = evaluate_retrieval_system(retriever_500, eval_questions)

comp_df = pd.DataFrame([metrics_300, metrics_500], index=["300-Token Chunks", "500-Token Chunks"])
print("\n=== RETRIEVAL BENCHMARK COMPARISON ===")
print(comp_df.to_string())

# Save comparison results
comp_df.to_csv(os.path.join(RESULTS_DIR, "retrieval_metrics.csv"))



Evaluating 300-Token Chunk Configuration...
Evaluating 500-Token Chunk Configuration...

=== RETRIEVAL BENCHMARK COMPARISON ===
                  Recall@1 (%)  Recall@5 (%)     MRR  Avg Latency (ms)
300-Token Chunks           0.0          6.67  0.0222             71.13
500-Token Chunks           0.0         13.33  0.0444             86.26


### 11. Rigorous Retrieval Failure Analysis
**What it does:** Scans the evaluation log for actual questions where target chunks were not ranked in the top position or failed Recall@5.
**Why it is required:** Identifies authentic edge cases to propose concrete architectural remedies.



In [15]:
failures = [d for d in details_300 if d["hit_rank"] != 1 or d["hit_rank"] is None]

print(f"Total Sub-optimal Queries in 300-token experiment: {len(failures)}")
for idx, f in enumerate(failures[:2]):
    print(f"\n--- Retrieval Analysis Case #{idx+1} ---")
    print(f"Query ID: {f['id']}")
    print(f"Question: {f['question']}")
    print(f"Expected Target Page: {f['target_page']}")
    print(f"Actual Rank Achieved: {f['hit_rank']}")
    print(f"Top Retrieved Page: {f['top_retrieved_page']}")
    print(f"Top Retrieved Snippet: {f['top_retrieved_text']}...")

    # Root cause diagnosis
    if f["hit_rank"] is None:
        cause = "Vocabulary mismatch or definition dispersed across page boundary."
        remedy = "Implement parent-document retrieval (small-to-big retrieval) or hypothetical document embeddings (HyDE)."
    else:
        cause = "Generic introductory terminology diluted dense semantic similarity."
        remedy = "Increase BM25 weight in hybrid fusion or apply contextual query expansion."

    print(f"Diagnostic Cause: {cause}")
    print(f"Proposed Fix: {remedy}")



Total Sub-optimal Queries in 300-token experiment: 15

--- Retrieval Analysis Case #1 ---
Query ID: Q1
Question: What are the four core functions of the NIST AI Risk Management Framework Core?
Expected Target Page: 20
Actual Rank Achieved: None
Top Retrieved Page: 25
Top Retrieved Snippet: nist ai 100 - 1 ai rmf 1. 0 part 2 : core and profiles 5. ai rmf core the ai rmf core provides outcomes and actions that enable dialogue, understandin...
Diagnostic Cause: Vocabulary mismatch or definition dispersed across page boundary.
Proposed Fix: Implement parent-document retrieval (small-to-big retrieval) or hypothetical document embeddings (HyDE).

--- Retrieval Analysis Case #2 ---
Query ID: Q2
Question: Which function in the NIST AI RMF Core is designed to be cross-cutting and infuses risk management across the other three functions?
Expected Target Page: 21
Actual Rank Achieved: None
Top Retrieved Page: 25
Top Retrieved Snippet: nist ai 100 - 1 ai rmf 1. 0 part 2 : core and profiles 5. ai r

### 12. Generative LLM Answer Synthesis with Explicit Citations
**What it does:** Synthesizes final answers using `Qwen/Qwen2.5-0.5B-Instruct` grounded strictly on retrieved passages with page-number citations.



In [16]:
from transformers import AutoModelForCausalLM

LLM_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading generator LLM: {LLM_ID}...")
gen_model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)
gen_tokenizer = AutoTokenizer.from_pretrained(LLM_ID, trust_remote_code=True)

def generate_grounded_answer(query, retrieved_chunks):
    context_str = ""
    for c in retrieved_chunks[:3]:
        context_str += f"[Page {c['page_number']} | Chunk {c['chunk_id']}]:\n{c['text']}\n\n"

    system_instruction = (
        "You are an authoritative NIST AI compliance assistant. "
        "Answer the user query strictly based on the provided context passages. "
        "Every claim must cite its source page in brackets, e.g. [Page X]. "
        "If the information is not present in the context, state 'Information not found in document.'"
    )

    user_prompt = f"Context Passages:\n{context_str}\nQuestion: {query}\nAnswer:"

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = gen_tokenizer(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    gen_tokens = out[0][inputs["input_ids"].shape[1]:]
    return gen_tokenizer.decode(gen_tokens, skip_special_tokens=True)

# Generate answer for demo question
demo_q = eval_questions[0]["question"]
top_passages = rerank_passages(demo_q, retriever_300.hybrid_search(demo_q, top_k=6), top_k=3)
generated_ans = generate_grounded_answer(demo_q, top_passages)

print(f"--- RAG GENERATION DEMO ---")
print(f"Question: {demo_q}")
print(f"\nSynthesized Answer:\n{generated_ans}")



Loading generator LLM: Qwen/Qwen2.5-0.5B-Instruct...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

--- RAG GENERATION DEMO ---
Question: What are the four core functions of the NIST AI Risk Management Framework Core?

Synthesized Answer:
The four core functions of the NIST AI Risk Management Framework Core are:

- Govern
- Map
- Measure
- Manage


### 13. Deep Conceptual Explanations

#### 1. What BM25 Adds to Embedding-Based Retrieval
Embedding models map queries and documents into a shared continuous semantic space. While effective at understanding conceptual similarity (e.g., recognizing that "adverse event" is related to "system malfunction"), embeddings struggle with exact lexical matches—such as specific acronyms ("TEVV"), section identifiers ("NIST AI 100-1 Section 3.2"), or precise nomenclature. BM25 (Best Matching 25) relies on term frequency-inverse document frequency ($TF\text{-}IDF$) saturation curves. By combining BM25 with dense vectors via Reciprocal Rank Fusion, the retriever achieves both lexical precision and semantic generalization.

#### 2. What a Reranker Does
Bi-encoder embedding retrievers encode queries and documents independently into fixed-dimensional vectors, evaluating similarity via dot products. This design is fast ($O(1)$ lookup via FAISS) but lacks cross-attention between query tokens and document tokens. A Cross-Encoder reranker feeds the concatenated `(Query, Passage)` pair through all Transformer self-attention layers simultaneously, computing token-level interactions. Applying the reranker as a secondary stage over the top 20 candidate passages provides high accuracy without the latency of scoring an entire corpus.

#### 3. Dense vs. Sparse Retrieval
- **Sparse Retrieval (e.g., BM25, SPLADE)**: Represents documents as high-dimensional, sparse vectors where each dimension corresponds to an exact dictionary term. Highly sensitive to vocabulary mismatch (synonym failure), but extremely efficient and zero-shot robust on out-of-domain technical terminology.
- **Dense Retrieval (e.g., all-MiniLM-L6-v2, Contriever)**: Represents documents as compact continuous vectors (384 to 1536 dimensions). Captures paraphrasing, semantics, and multilingual alignment, but can suffer from "semantic drift" on out-of-domain jargon.

#### 4. Why Chunk Size Matters
- **Small Chunks (e.g., 100 - 300 tokens)**: Granular, highly focused representations that produce sharp cosine similarity matches with specific questions. However, they risk fragmenting necessary surrounding context (e.g., separating an introductory premise from its concluding condition).
- **Large Chunks (e.g., 500 - 1000 tokens)**: Preserve complete paragraphs and holistic ideas, reducing context fragmentation. However, large chunks dilute dense vector specificity (the "embedding averaging" effect) and consume larger LLM prompt token budgets.

#### 5. Why Overlap is Useful
Document chunking inevitably creates arbitrary boundaries that can split a sentence or logical argument in half. An overlap window (e.g., 50 tokens) ensures that every boundary region appears unbroken in at least one chunk, preventing retrieval failures caused by split terminology.

#### 6. Why Retrieval Accuracy and Answer Quality are Different
- **Retrieval Accuracy (Recall, MRR)** measures whether the correct text segments are placed inside the prompt context.
- **Answer Quality (Factual Correctness, Faithfulness, Fluency)** depends on the generator LLM's comprehension, instruction-following ability, and context window utilization. A system can have 100% Recall@5 yet produce hallucinations if the LLM misinterprets the retrieved context or relies on its pre-trained parametric memory instead of the grounded passages.



### 14. Export Task 2 Artifacts & Results Archive
**What it does:** Archives the index files, extracted page text, evaluation metrics, and query logs into a portable ZIP package.



In [17]:
import shutil

zip_filename = os.path.join(BASE_DIR, "Task_2_RAG_Results.zip")
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', BASE_DIR)
print(f"Task 2 Results archived successfully: {zip_filename}")

try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Archive ready for manual download.")



Task 2 Results archived successfully: /content/LLM-Engineering-Assignment/Task-2-RAG/Task_2_RAG_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>